#task 1

In [ ]:


import os
from dotenv import load_dotenv  
load_dotenv()

from langchain_groq import ChatGroq



llm = ChatGroq(
    model="llama3-70b-8192",
    temperature=0
)

print(" ChatGroq initialized successfully")

#task2

In [ ]:

response = llm.invoke("Explain Artificial Intelligence in one sentence")

print(" Response:")
print(response.content)

#task 3

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


loader = TextLoader("sample.txt")
documents = loader.load()


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print(f" Total chunks created: {len(docs)}")
print("\nSample chunk:\n", docs[0].page_content)

#task 4

In [ ]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS


embeddings = OpenAIEmbeddings()


vectorstore = FAISS.from_documents(docs, embeddings)


retriever = vectorstore.as_retriever()

print(" Vector store created and retriever ready")


query = "What is Artificial Intelligence?"
results = retriever.get_relevant_documents(query)

print("\n Top Result:\n", results[0].page_content)

#task 5

In [ ]:
from langchain.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template("""
You are an AI assistant.

Answer the question ONLY using the provided context.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
""")

print(" RAG Prompt Template created")

#task 6

In [ ]:
from langchain.schema.runnable import RunnablePassthrough


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

print(" RAG Chain created")

questions = [
    "What is Artificial Intelligence?",
    "Explain machine learning briefly"
]

for q in questions:
    response = rag_chain.invoke(q)
    print(f"\n Question: {q}")
    print(f" Answer: {response.content}")

#task 7

In [ ]:


import streamlit as st

st.title(" ChatGroq RAG Chatbot")


if "messages" not in st.session_state:
    st.session_state.messages = []


uploaded_file = st.file_uploader("Upload a file (TXT/PDF)", type=["txt", "pdf"])

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])


user_input = st.chat_input("Ask a question...")

if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})

    with st.chat_message("user"):
        st.markdown(user_input)

#task 8

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

if uploaded_file:
   
    with open("temp.txt", "wb") as f:
        f.write(uploaded_file.read())

    
    loader = TextLoader("temp.txt")
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    split_docs = splitter.split_documents(docs)

   
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(split_docs, embeddings)
    retriever = vectorstore.as_retriever()

    
    if user_input:
        response = rag_chain.invoke(user_input)

        st.session_state.messages.append({"role": "assistant", "content": response.content})

        with st.chat_message("assistant"):
            st.markdown(response.content)

#task 9

In [ ]:
chat_history = []

def ask_question(question):
    response = rag_chain.invoke(question)
    chat_history.append((question, response.content))
    return response.content


q1 = "What is the document about?"
a1 = ask_question(q1)

q2 = "Summarize it in 2 lines"
a2 = ask_question(q2)


q3 = "Who is the President of the USA?"
a3 = ask_question(q3)

for q, a in chat_history:
    print(f"\n User: {q}")
    print(f" Bot: {a}")

print("\n Verification:")
print("- Answers should come only from document context")
print("- Out-of-context should return: 'I don't know'")
print("- Chat history maintained in list")

#task 10

In [ ]:
import streamlit as st

st.title(" ChatGroq RAG App")

if "messages" not in st.session_state:
    st.session_state.messages = []

if "rag_chain" not in st.session_state:
    st.session_state.rag_chain = None

uploaded_file = st.file_uploader("Upload TXT file", type=["txt"])

try:
    if uploaded_file:
        with open("temp.txt", "wb") as f:
            f.write(uploaded_file.read())

        from langchain.document_loaders import TextLoader
        from langchain.text_splitter import RecursiveCharacterTextSplitter
        from langchain.embeddings.openai import OpenAIEmbeddings
        from langchain.vectorstores import FAISS

        loader = TextLoader("temp.txt")
        docs = loader.load()

        splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        split_docs = splitter.split_documents(docs)

        embeddings = OpenAIEmbeddings()
        vectorstore = FAISS.from_documents(split_docs, embeddings)
        retriever = vectorstore.as_retriever()

        from langchain.schema.runnable import RunnablePassthrough

        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)

        st.session_state.rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt
            | llm
        )

    
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    user_input = st.chat_input("Ask something...")

    if user_input and st.session_state.rag_chain:
        st.session_state.messages.append({"role": "user", "content": user_input})

        with st.chat_message("user"):
            st.markdown(user_input)

        response = st.session_state.rag_chain.invoke(user_input)

        st.session_state.messages.append({"role": "assistant", "content": response.content})

        with st.chat_message("assistant"):
            st.markdown(response.content)

except Exception as e:
    st.error(f"Error: {str(e)}")

print(" App supports dynamic loading, fast Groq responses, memory & error handling")

#task 11

*1. Why Groq is appropriate for RAG chatbots*
Extremely low latency inference is offered by Groq. This means that chatbot responses are given almost in real-time.

*2. What is the difference between Groq RAG and OpenAI RAG?*
- Pros of Groq RAG: Fast inference, low latency. Suitable for real-time applications.
- Pros of OpenAI RAG: Better reliability, variety of models.
- Trade-off: Groq = speed , OpenAI = reliability + features 

*3. What is the role of Streamlit in quick GenAI development?*
Streamlit is used to create interactive UIs quickly without needing to write any frontend code. This helps in quick testing and deployment of GenAI applications such as chatbots.